In [61]:
import pandas as pd
from sqlalchemy import create_engine

In [62]:
# File Path
file_path = 'C:\_Data_Analyst\_Projects_\Project_1\Raw_Data_online_retail_II.csv'

In [63]:
# Exploration Data
df = pd.read_csv(file_path)
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [64]:
# Exploration Data Before Data Cleaning

Total_Count_DB = df.count().sum()
Unique_Count_DB = df.nunique()
Rows_DB, Columns_D = df.shape
Null_Count_DB = df.isnull().sum().sum()
Duplicate_Count_DB = df.duplicated().sum()
Data_Types_DB = df.dtypes
MissingValue_Percentage = (Null_Count_DB/Total_Count_DB)*100

In [65]:
print(f"""
========================================================
             DATASET OVERVIEW
========================================================

Data Types :
{Data_Types_DB}

Total Count :
{Total_Count_DB}

Unique Count :
{Unique_Count_DB}

Dataset Shape (Rows, Columns) :
({Rows_DB}, {Columns_D})

Null Count :
{Null_Count_DB}

Duplicate Count :
{Duplicate_Count_DB}

Missing Value Percentage:
{MissingValue_Percentage:,.2f}%

========================================================
""")


             DATASET OVERVIEW

Data Types :
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

Total Count :
8291579

Unique Count :
Invoice        53628
StockCode       5305
Description     5698
Quantity        1057
InvoiceDate    47635
Price           2807
Customer ID     5942
Country           43
dtype: int64

Dataset Shape (Rows, Columns) :
(1067371, 8)

Null Count :
247389

Duplicate Count :
34335

Missing Value Percentage:
2.98%




In [66]:
# Remove duplicates
df = df.drop_duplicates()

In [67]:
# Remove Empty Columns
df = df.dropna()

In [68]:
# Convert InvoiceDate to Datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [69]:
# Correct Data Types
df = df.convert_dtypes()

In [70]:
# Investigate Negative Values and Price Equal to Zero
Negative_Quantity_Records  = df[df['Quantity'] < 0]
Zero_Price_Records   = df[df['Price']==0]
Num_Negative_Quantity  = Negative_Quantity_Records['Quantity'].count()
Num_Zero_Price  = Zero_Price_Records['Price'].count()
Summary_Stats =df[["Quantity", "Price"]].describe()

In [71]:
print(f"""
========================================================
           NEGATIVE & ZERO VALUE ANALYSIS
========================================================

Number of Negative Quantity Records :
{Num_Negative_Quantity}

Number of Zero Price Records :
{Num_Zero_Price}

Summary Statistics :
{Summary_Stats}

Negative Quantity Records :
{Negative_Quantity_Records[['Invoice', 'StockCode', 'Description', 'Quantity']]}

Zero Price Records :
{Zero_Price_Records[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']]}

========================================================
""")


           NEGATIVE & ZERO VALUE ANALYSIS

Number of Negative Quantity Records :
18390

Number of Zero Price Records :
70

Summary Statistics :
         Quantity      Price
count    797885.0   797885.0
mean     12.60298   3.702732
std    191.670371  71.392549
min      -80995.0        0.0
25%           2.0       1.25
50%           5.0       1.95
75%          12.0       3.75
max       80995.0    38970.0

Negative Quantity Records :
         Invoice StockCode                       Description  Quantity
178      C489449     22087          PAPER BUNTING WHITE LACE       -12
179      C489449    85206A      CREAM FELT EASTER EGG BASKET        -6
180      C489449     21895     POTTING SHED SOW 'N' GROW SET        -4
181      C489449     21896                POTTING SHED TWINE        -6
182      C489449     22083        PAPER CHAIN KIT RETRO SPOT       -12
...          ...       ...                               ...       ...
1065910  C581490     23144   ZINC T-LIGHT HOLDER STARS SMALL       -

In [72]:
# Cancellation Analysis
Cancelled_Transactions  = df[df['Invoice'].astype(str).str.startswith("C")]
Num_Cancelled_Transactions = Cancelled_Transactions.shape[0]
Total_Cancelled_Quantity =Cancelled_Transactions['Quantity'].abs().sum()

In [73]:
print(f"""
========================================================
              CANCELLATION ANALYSIS
========================================================

Number of Cancelled Transactions :
{Num_Cancelled_Transactions}

Total Cancelled Quantity :
{Total_Cancelled_Quantity}

Cancelled Transactions :
{Cancelled_Transactions[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']]}

========================================================
""")


              CANCELLATION ANALYSIS

Number of Cancelled Transactions :
18390

Total Cancelled Quantity :
472976

Cancelled Transactions :
         Invoice StockCode                       Description  Quantity   Price
178      C489449     22087          PAPER BUNTING WHITE LACE       -12    2.95
179      C489449    85206A      CREAM FELT EASTER EGG BASKET        -6    1.65
180      C489449     21895     POTTING SHED SOW 'N' GROW SET        -4    4.25
181      C489449     21896                POTTING SHED TWINE        -6     2.1
182      C489449     22083        PAPER CHAIN KIT RETRO SPOT       -12    2.95
...          ...       ...                               ...       ...     ...
1065910  C581490     23144   ZINC T-LIGHT HOLDER STARS SMALL       -11    0.83
1067002  C581499         M                            Manual        -1  224.69
1067176  C581568     21258        VICTORIAN SEWING BOX LARGE        -5   10.95
1067177  C581569     84978  HANGING HEART JAR T-LIGHT HOLDER        -1

In [74]:
# Create Derived Features
df['Revenue'] = df['Quantity']*df['Price']
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month_name()
df['Month_Num'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day_name()
df['Day_Num'] = df['InvoiceDate'].dt.dayofweek
df['Quarter'] = 'Q' + df['InvoiceDate'].dt.quarter.astype(str)
df["TransactionType"] = df["Invoice"].astype(str).apply(
    lambda x: "cancellation" if x.startswith("C") else "Sale"
)

In [75]:
# Validate Cleaned Dataset

Total_Count_After_Cleaning = df["Invoice"].count()
Unique_Count_After_Cleaning = df.nunique()
Rows_After_Cleaning, Columns_After_Cleaning =  df.shape
Null_Count_After_Cleaning = df.isnull().sum()
Duplicate_Count_After_Cleaning = df.duplicated().sum()
Data_Types_After_Cleaning = df.dtypes
TransactionType_Count = df["TransactionType"].value_counts()

In [76]:
print(f"""
========================================================
         DATASET VALIDATION AFTER CLEANING
========================================================

Total Count After Cleaning     :
{Total_Count_After_Cleaning}

Transaction Type Count         :
{TransactionType_Count}

Dataset Shape (Rows, Columns)  :
({Rows_After_Cleaning}, {Columns_After_Cleaning})

Null Count After Cleaning      :
{Null_Count_After_Cleaning}

Unique Count After Cleaning    :
{Unique_Count_After_Cleaning}

Duplicate Count After Cleaning :
{Duplicate_Count_After_Cleaning}

Data Types After Cleaning      :
{Data_Types_After_Cleaning}

========================================================
""")


         DATASET VALIDATION AFTER CLEANING

Total Count After Cleaning     :
797885

Transaction Type Count         :
TransactionType
Sale            779495
cancellation     18390
Name: count, dtype: int64

Dataset Shape (Rows, Columns)  :
(797885, 16)

Null Count After Cleaning      :
Invoice            0
StockCode          0
Description        0
Quantity           0
InvoiceDate        0
Price              0
Customer ID        0
Country            0
Revenue            0
Year               0
Month              0
Month_Num          0
Day                0
Day_Num            0
Quarter            0
TransactionType    0
dtype: int64

Unique Count After Cleaning    :
Invoice            44876
StockCode           4646
Description         5299
Quantity             643
InvoiceDate        41439
Price               1022
Customer ID         5942
Country               41
Revenue             5625
Year                   3
Month                 12
Month_Num             12
Day                    7
Day_

In [77]:
# Preview Dataset After Adding Derived Columns
print(f"""Preview Dataset After Adding Derived Columns:
{df}""")

Preview Dataset After Adding Derived Columns:
        Invoice StockCode                          Description  Quantity  \
0        489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434    79323P                   PINK CHERRY LIGHTS        12   
2        489434    79323W                  WHITE CHERRY LIGHTS        12   
3        489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...       ...                                  ...       ...   
1067366  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587      POST                              POSTAGE         1   

                InvoiceDate  Price  Custo

In [78]:
# Business KPIs

Total_Order_Count = df['Invoice'].nunique()
Total_Sales_Order_Count = df[df['TransactionType'] == "Sale"]['Invoice'].nunique()
Total_Cancelled_Order_Count = df[df['TransactionType'] == "cancellation"]['Invoice'].nunique()
Total_Revenue = df[df['TransactionType']=='Sale']['Revenue'].sum()
Total_Sales_Quantity = df[df['TransactionType'] == "Sale"]['Quantity'].sum()
Total_Return_Amount = df[df['TransactionType'] == "cancellation"]['Revenue'].abs().sum()
Total_Return_Quantity = df[df['TransactionType'] == "cancellation"]['Quantity'].abs().sum()
Country_Count = df['Country'].nunique()
Customer_Count = df['Customer ID'].nunique()
Stock_Count = df['StockCode'].nunique()
Return_Rate = (Total_Return_Quantity/Total_Sales_Quantity)*100

In [79]:
print(f"""
===========================
      DATASET SUMMARY
===========================

Total Orders               : {Total_Order_Count :,}
Sales Orders               : {Total_Sales_Order_Count:,}
Cancelled Orders           : {Total_Cancelled_Order_Count:,}

Total Revenue              : {Total_Revenue:,.2f}
Total Return Amount        : {Total_Return_Amount:,.2f}

Total Sales Quantity       : {Total_Sales_Quantity:,}
Total_Return_Quantity      : {Total_Return_Quantity:,}

Countries                  : {Country_Count:,}
Customers                  : {Customer_Count:,}
Unique Stock Codes         : {Stock_Count:,}

Return Rate                : {Return_Rate:,.2f}%

===========================
""")


      DATASET SUMMARY

Total Orders               : 44,876
Sales Orders               : 36,975
Cancelled Orders           : 7,901

Total Revenue              : 17,374,804.27
Total Return Amount        : 1,084,812.98

Total Sales Quantity       : 10,528,705
Total_Return_Quantity      : 472,976

Countries                  : 41
Customers                  : 5,942
Unique Stock Codes         : 4,646

Return Rate                : 4.49%




In [80]:
# Sales Quantity Analysis
Sales_Quantity_By_Year = df[df['TransactionType']=='Sale'].groupby('Year')['Quantity'].sum().sort_values(ascending=False)
Sales_Quantity_By_Month = df[df['TransactionType']=='Sale'].groupby(['Month_Num','Month'])['Quantity'].sum().sort_index(level=0)
Sales_Quantity_By_Weekday = df[df['TransactionType']=='Sale'].groupby(['Day_Num','Day'])['Quantity'].sum().sort_index(level=0)
Sales_Quantity_By_Quarter = df[df['TransactionType']=='Sale'].groupby('Quarter')['Quantity'].sum().sort_values(ascending=False)

# Revenue Analysis
Revenue_By_Year = df[df['TransactionType']=='Sale'].groupby('Year')['Revenue'].sum().sort_values(ascending=False)
Revenue_By_Month = df[df['TransactionType']=='Sale'].groupby(['Month_Num','Month'])['Revenue'].sum().sort_index(level=0)
Revenue_By_Weekday = df[df['TransactionType']=='Sale'].groupby(['Day_Num','Day'])['Revenue'].sum().sort_index(level=0)
Revenue_By_Quarter = df[df['TransactionType']=='Sale'].groupby('Quarter')['Revenue'].sum().sort_values(ascending=False)

# Return Quantity Analysis
Return_Quantity_By_Year = df[df['TransactionType']=='cancellation'].groupby('Year')['Quantity'].sum().abs()
Return_Quantity_By_Month = df[df['TransactionType']=='cancellation'].groupby(['Month_Num','Month'])['Quantity'].sum().abs().sort_index(level=0)
Return_Quantity_By_Weekday  = df[df['TransactionType']=='cancellation'].groupby(['Day_Num','Day'])['Quantity'].sum().abs().sort_index(level=0)
Return_Quantity_By_Quarter = df[df['TransactionType']=='cancellation'].groupby('Quarter')['Quantity'].sum().abs()

# Return Revenue Analysis
Return_Amount_By_Year = df[df['TransactionType']=='cancellation'].groupby('Year')['Revenue'].sum().abs()
Return_Amount_By_Month= df[df['TransactionType']=='cancellation'].groupby(['Month_Num','Month'])['Revenue'].sum().abs().sort_index(level=0)
Return_Amount_By_Weekday= df[df['TransactionType']=='cancellation'].groupby(['Day_Num','Day'])['Revenue'].sum().abs().sort_index(level=0)
Return_Amount_By_Quarter= df[df['TransactionType']=='cancellation'].groupby('Quarter')['Revenue'].sum().abs()

In [81]:
print(f"""
========================================================
             SALES & RETURN ANALYSIS SUMMARY
========================================================

SALES QUANTITY BY YEAR
----------------------
{Sales_Quantity_By_Year}

SALES QUANTITY BY MONTH
-----------------------
{Sales_Quantity_By_Month}

SALES QUANTITY BY WEEKDAY
-------------------------
{Sales_Quantity_By_Weekday}

SALES QUANTITY BY QUARTER
-------------------------
{Sales_Quantity_By_Quarter}

========================================================

REVENUE BY YEAR
---------------
{Revenue_By_Year}

REVENUE BY MONTH
----------------
{Revenue_By_Month}

REVENUE BY WEEKDAY
------------------
{Revenue_By_Weekday}

REVENUE BY QUARTER
------------------
{Revenue_By_Quarter}

========================================================

RETURN QUANTITY BY YEAR
-----------------------
{Return_Quantity_By_Year}

RETURN QUANTITY BY MONTH
------------------------
{Return_Quantity_By_Month}

RETURN QUANTITY BY WEEKDAY
--------------------------
{Return_Quantity_By_Weekday}

RETURN QUANTITY BY QUARTER
--------------------------
{Return_Quantity_By_Quarter}

========================================================

RETURN AMOUNT BY YEAR
---------------------
{Return_Amount_By_Year}

RETURN AMOUNT BY MONTH
----------------------
{Return_Amount_By_Month}

RETURN AMOUNT BY WEEKDAY
------------------------
{Return_Amount_By_Weekday}

RETURN AMOUNT BY QUARTER
------------------------
{Return_Amount_By_Quarter}

========================================================
""")


             SALES & RETURN ANALYSIS SUMMARY

SALES QUANTITY BY YEAR
----------------------
Year
2010    5275173
2011    4854824
2009     398708
Name: Quantity, dtype: Int64

SALES QUANTITY BY MONTH
-----------------------
Month_Num  Month    
1          January       718650
2          February      636914
3          March         849724
4          April         641973
5          May           757908
6          June          752894
7          July          692010
8          August        850741
9          September    1111460
10         October      1188048
11         November     1331834
12         December      996549
Name: Quantity, dtype: Int64

SALES QUANTITY BY WEEKDAY
-------------------------
Day_Num  Day      
0        Monday       1837616
1        Tuesday      1987235
2        Wednesday    1891981
3        Thursday     2227620
4        Friday       1560352
5        Saturday        5119
6        Sunday       1018782
Name: Quantity, dtype: Int64

SALES QUANTITY BY QUARTER
----

In [82]:
# Customer Analysis
Top_5_Customers = df[df['TransactionType']=='Sale'].groupby('Customer ID')['Revenue'].sum().nlargest(10)
Top_10_Customer_Order_Frequency = df[df['TransactionType']=='Sale'].groupby('Customer ID')['Invoice'].nunique().nlargest(10)

In [83]:
print(f"""
=========================================
        CUSTOMER ANALYSIS
=========================================

Top 5 Customers (by Revenue)
----------------------------
{Top_5_Customers}

Top 10 Customers by Order Frequency
-----------------------------------
{Top_10_Customer_Order_Frequency}

=========================================
""")


        CUSTOMER ANALYSIS

Top 5 Customers (by Revenue)
----------------------------
Customer ID
18102    580987.04
14646    528602.52
14156    313437.62
14911    291420.81
17450    244784.25
13694    195640.69
17511    172132.87
16446     168472.5
16684    147142.77
12415    144458.37
Name: Revenue, dtype: Float64

Top 10 Customers by Order Frequency
-----------------------------------
Customer ID
14911    398
12748    337
17841    211
15311    208
13089    203
14606    192
14156    156
17850    155
14646    152
18102    145
Name: Invoice, dtype: int64




In [84]:
# Product Performance Analysis

Top_10_Products = df[df['TransactionType']=='Sale'].groupby('Description')['Quantity'].sum().nlargest(10)
Bottom_10_Products = df[df['TransactionType']=='Sale'].groupby('Description')['Quantity'].sum().nsmallest(10)

Top_10_Products_Revenue  = df[df['TransactionType']=='Sale'].groupby('Description')['Revenue'].sum().nlargest(10)
Bottom_10_Products_Revenue  = df[df['TransactionType']=='Sale'].groupby('Description')['Revenue'].sum().nsmallest(10)

Most_Frequent_Products  = df[df['TransactionType']=='Sale'].groupby('Description')['Invoice'].nunique().sort_values(ascending=False).nlargest(10)
Average_Product_Price  = df[df['TransactionType']=='Sale'].groupby('Description')['Price'].mean().sort_values(ascending=False).nlargest(10)


Most_Returned_Products  = df[df['TransactionType']=='cancellation'].groupby('Description')['Quantity'].sum().sort_values().abs().nlargest(10)


Product_Performance_Year  = df[df['TransactionType']== 'Sale'].groupby(['Year', 'Description'])['Quantity'].sum()

In [85]:
print(f"""
========================================================
              PRODUCT PERFORMANCE ANALYSIS
========================================================

TOP 10 PRODUCTS BY SALES QUANTITY
---------------------------------
{Top_10_Products}

BOTTOM 10 PRODUCTS BY SALES QUANTITY
------------------------------------
{Bottom_10_Products}

TOP 10 PRODUCTS BY REVENUE
--------------------------
{Top_10_Products_Revenue}

BOTTOM 10 PRODUCTS BY REVENUE
-----------------------------
{Bottom_10_Products_Revenue}

MOST FREQUENTLY PURCHASED PRODUCTS
----------------------------------
{Most_Frequent_Products}

TOP 10 PRODUCTS BY AVERAGE PRICE
--------------------------------
{Average_Product_Price}

MOST RETURNED PRODUCTS
----------------------
{Most_Returned_Products}

PRODUCT PERFORMANCE BY YEAR
---------------------------
{Product_Performance_Year}

========================================================
""")


              PRODUCT PERFORMANCE ANALYSIS

TOP 10 PRODUCTS BY SALES QUANTITY
---------------------------------
Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS     105185
WHITE HANGING HEART T-LIGHT HOLDER     91757
PAPER CRAFT , LITTLE BIRDIE            80995
ASSORTED COLOUR BIRD ORNAMENT          78234
MEDIUM CERAMIC TOP STORAGE JAR         77916
JUMBO BAG RED RETROSPOT                74224
BROCADE RING PURSE                     70082
PACK OF 60 PINK PAISLEY CAKE CASES     54592
60 TEATIME FAIRY CAKE CASES            52828
PACK OF 72 RETRO SPOT CAKE CASES       45129
Name: Quantity, dtype: Int64

BOTTOM 10 PRODUCTS BY SALES QUANTITY
------------------------------------
Description
 I LOVE LONDON MINI RUCKSACK        1
6 HOOK JEWEL STAND LILAC DRESS      1
AMBER CRYSTAL DROP EARRINGS         1
BAROQUE BUTTERFLY EARRINGS RED      1
BISCUIT TIN, MINT,IVORY, VINTAGE    1
BLACK DIAMOND CLUSTER EARRINGS      1
BLACK RND BULLET"KEEP CLEAN" BIN    1
BOX OF 3 PEBBLE CANDLES             1
CANDY

In [86]:
# Sales Quantity Analysis by Country

Sales_Quantity_By_Country = df[df['TransactionType']=='Sale'].groupby('Country')['Quantity'].sum().sort_values(ascending=False)
Sales_Quantity_By_Country_Year = df[df['TransactionType']=='Sale'].groupby(['Country','Year'])['Quantity'].sum()
Sales_Quantity_By_Country_Month = df[df['TransactionType']=='Sale'].groupby(['Country','Month_Num','Month'])['Quantity'].sum().sort_index(level=0)
Sales_Quantity_By_Country_Day = df[df['TransactionType']=='Sale'].groupby(['Country','Day_Num','Day'])['Quantity'].sum().sort_index(level=0)
Sales_Quantity_By_Country_Quarter = df[df['TransactionType']=='Sale'].groupby(['Country','Quarter'])['Quantity'].sum()


# Revenue Analysis

Revenue_By_Country = df[df['TransactionType']=='Sale'].groupby('Country')['Revenue'].sum().sort_values(ascending=False)
Revenue_By_Country_Year = df[df['TransactionType']=='Sale'].groupby(['Country','Year'])['Revenue'].sum()
Revenue_By_Country_Month = df[df['TransactionType']=='Sale'].groupby(['Country','Month_Num','Month'])['Revenue'].sum().sort_index(level=0)
Revenue_By_Country_Day = df[df['TransactionType']=='Sale'].groupby(['Country','Day_Num','Day'])['Revenue'].sum().sort_index(level=0)
Revenue_By_Country_Quarter = df[df['TransactionType']=='Sale'].groupby(['Country','Quarter'])['Revenue'].sum()


# Return Quantity Analysis

Return_Quantity_By_Country = df[df['TransactionType']=='cancellation'].groupby('Country')['Quantity'].sum().abs().sort_values(ascending=False)
Return_Quantity_By_Country_Year = df[df['TransactionType']=='cancellation'].groupby(['Country','Year'])['Quantity'].sum().abs()
Return_Quantity_By_Country_Month = df[df['TransactionType']=='cancellation'].groupby(['Country','Month_Num','Month'])['Quantity'].sum().abs().sort_index(level=0)
Return_Quantity_By_Country_Day = df[df['TransactionType']=='cancellation'].groupby(['Country','Day_Num','Day'])['Quantity'].sum().abs().sort_index(level=0)
Return_Quantity_By_Country_Quarter = df[df['TransactionType']=='cancellation'].groupby(['Country','Quarter'])['Quantity'].sum().abs()


# Return Amount Analysis

Return_Amount_By_Country = df[df['TransactionType']=='cancellation'].groupby('Country')['Revenue'].sum().abs().sort_values(ascending=False)
Return_Amount_By_Country_Year = df[df['TransactionType']=='cancellation'].groupby(['Country','Year'])['Revenue'].sum().abs()
Return_Amount_By_Country_Month = df[df['TransactionType']=='cancellation'].groupby(['Country','Month_Num','Month'])['Revenue'].sum().abs().sort_index(level=0)
Return_Amount_By_Country_Day = df[df['TransactionType']=='cancellation'].groupby(['Country','Day_Num','Day'])['Revenue'].sum().abs().sort_index(level=0)
Return_Amount_By_Country_Quarter = df[df['TransactionType']=='cancellation'].groupby(['Country','Quarter'])['Revenue'].sum().abs()

In [87]:
print(f"""
========================================================
        COUNTRY-WISE SALES & RETURN ANALYSIS
========================================================

SALES QUANTITY BY COUNTRY
-------------------------
{Sales_Quantity_By_Country}

SALES QUANTITY BY COUNTRY & YEAR
--------------------------------
{Sales_Quantity_By_Country_Year}

SALES QUANTITY BY COUNTRY & MONTH
---------------------------------
{Sales_Quantity_By_Country_Month}

SALES QUANTITY BY COUNTRY & DAY
-------------------------------
{Sales_Quantity_By_Country_Day}

SALES QUANTITY BY COUNTRY & QUARTER
-----------------------------------
{Sales_Quantity_By_Country_Quarter}

========================================================

REVENUE BY COUNTRY
------------------
{Revenue_By_Country}

REVENUE BY COUNTRY & YEAR
-------------------------
{Revenue_By_Country_Year}

REVENUE BY COUNTRY & MONTH
--------------------------
{Revenue_By_Country_Month}

REVENUE BY COUNTRY & DAY
------------------------
{Revenue_By_Country_Day}

REVENUE BY COUNTRY & QUARTER
----------------------------
{Revenue_By_Country_Quarter}

========================================================

RETURN QUANTITY BY COUNTRY
--------------------------
{Return_Quantity_By_Country}

RETURN QUANTITY BY COUNTRY & YEAR
---------------------------------
{Return_Quantity_By_Country_Year}

RETURN QUANTITY BY COUNTRY & MONTH
----------------------------------
{Return_Quantity_By_Country_Month}

RETURN QUANTITY BY COUNTRY & DAY
--------------------------------
{Return_Quantity_By_Country_Day}

RETURN QUANTITY BY COUNTRY & QUARTER
------------------------------------
{Return_Quantity_By_Country_Quarter}

========================================================

RETURN AMOUNT BY COUNTRY
------------------------
{Return_Amount_By_Country}

RETURN AMOUNT BY COUNTRY & YEAR
-------------------------------
{Return_Amount_By_Country_Year}

RETURN AMOUNT BY COUNTRY & MONTH
--------------------------------
{Return_Amount_By_Country_Month}

RETURN AMOUNT BY COUNTRY & DAY
------------------------------
{Return_Amount_By_Country_Day}

RETURN AMOUNT BY COUNTRY & QUARTER
----------------------------------
{Return_Amount_By_Country_Quarter}

========================================================
""")


        COUNTRY-WISE SALES & RETURN ANALYSIS

SALES QUANTITY BY COUNTRY
-------------------------
Country
United Kingdom          8545921
Netherlands              384519
EIRE                     318271
France                   270289
Denmark                  237471
Germany                  225173
Australia                104067
Sweden                    88495
Switzerland               52228
Spain                     50318
Belgium                   34778
Japan                     31643
Portugal                  27423
Norway                    23623
Channel Islands           21396
Italy                     15312
Finland                   14375
Austria                   11578
Cyprus                    10950
Greece                     7724
Singapore                  6994
United Arab Emirates       5839
Poland                     5688
USA                        5264
Israel                     5175
Unspecified                5113
Canada                     3657
Iceland                    29

In [88]:
# StockCode Analysis

Number_of_StockCode = df['StockCode'].nunique()
Number_of_Non_Numerical_StockCode = df.loc[~df['StockCode'].astype(str).str.isdecimal(),'StockCode'].nunique()
Non_Numerical_StockCode = df.loc[~df['StockCode'].astype(str).str.isdecimal(), 'StockCode'].unique()


In [89]:
print(f"""
========================================================
                 STOCKCODE ANALYSIS
========================================================

Number of Unique StockCodes :
{Number_of_StockCode}

Number of Non-Numerical StockCodes :
{Number_of_Non_Numerical_StockCode}

Non-Numerical StockCodes :
{Non_Numerical_StockCode}

========================================================
""")


                 STOCKCODE ANALYSIS

Number of Unique StockCodes :
4646

Number of Non-Numerical StockCodes :
1315

Non-Numerical StockCodes :
<StringArray>
['79323P', '79323W', '48173C', '35004B', '84596F', '84596L', '84507B',
 '84970S', '84031A', '84031B',
 ...
 '90141D', '84857B', '84387A', '16169E', '90176E',    'DOT',   'CRUK',
 '35819P', '90012A', '79157V']
Length: 1315, dtype: string




In [90]:
# Order Analysis
Orders_By_Year  = df.groupby('Year')['Invoice'].nunique()
Orders_By_Month = df.groupby(['Month_Num','Month'])['Invoice'].nunique().sort_index(level=0)
Orders_By_Quarter  = df.groupby('Quarter')['Invoice'].nunique()
AOV = Total_Revenue/Total_Sales_Order_Count

In [91]:
print(f"""
========================================================
          REVENUE & AVERAGE ORDER VALUE
========================================================

Total Revenue by Year
---------------------
{Orders_By_Year}

Total Revenue by Month
----------------------
{Orders_By_Month}

Total Revenue by Quarter
------------------------
{Orders_By_Quarter}

Average Order Value (AOV)
-------------------------
{AOV:.2f}

========================================================
""")


          REVENUE & AVERAGE ORDER VALUE

Total Revenue by Year
---------------------
Year
2009     1900
2010    22494
2011    20482
Name: Invoice, dtype: int64

Total Revenue by Month
----------------------
Month_Num  Month    
1          January      2532
2          February     2537
3          March        3526
4          April        2999
5          May          3617
6          June         3540
7          July         3306
8          August       3091
9          September    4119
10         October      4849
11         November     6231
12         December     4529
Name: Invoice, dtype: int64

Total Revenue by Quarter
------------------------
Quarter
Q1     8595
Q2    10156
Q3    10516
Q4    15609
Name: Invoice, dtype: int64

Average Order Value (AOV)
-------------------------
469.91




In [92]:
# Outlier Analysis - Summary Statistics
Sales_Summary_Statistics = df.loc[df['TransactionType']=='Sale',['Quantity','Price','Revenue']].describe()

In [93]:
print(f"""
========================================================
              SALES SUMMARY STATISTICS
========================================================

Sales Summary Statistics
------------------------
{Sales_Summary_Statistics}

========================================================
""")


              SALES SUMMARY STATISTICS

Sales Summary Statistics
------------------------
         Quantity      Price     Revenue
count    779495.0   779495.0    779495.0
mean    13.507085   3.218199   22.289821
std    146.540284  29.674823  227.416962
min           1.0        0.0         0.0
25%           2.0       1.25        4.95
50%           6.0       1.95       12.48
75%          12.0       3.75        19.8
max       80995.0    10953.5    168469.6




In [94]:
# # Database Configuration
USER = 'root'
HOST = '127.0.0.1'
PORT = '3306'
DataBase = 'online_retail_db'
Table_Name = 'online_retail_table'

In [95]:
# Export Dataset to MySQL
engine = None
try:
    engine = create_engine(f"mysql+pymysql://{USER}@{HOST}:{PORT}/{DataBase}")
    df.to_sql(
        Table_Name,
        engine,
        if_exists = 'replace',
        index = False,
        chunksize = 10000
    )
    
    print(f"'{Table_Name}' loaded successfully into '{DataBase}' database.")
    
except Exception as e:
    print("Failed to load data into MySQL")
    print(f"Error : {e}")
finally:
    if engine:
        engine.dispose()
        print("Data Connection Close")

'online_retail_table' loaded successfully into 'online_retail_db' database.
Data Connection Close


In [96]:
# Cross Verification
Total_Row_Cross_Check = df.shape[0]
Data_Types_Cross_Check = df.dtypes
Transaction_Count_Cross_Check = df.groupby('TransactionType')['Invoice'].nunique()
Summary_Statistics_Cross_Check = df.groupby(['TransactionType'])[['Quantity','Price','Revenue']].sum().abs()

In [97]:
print(f"""
========================================================
              DATABASE CROSS VERIFICATION
========================================================

Total Rows:
{Total_Row_Cross_Check}

--------------------------------------------------------
Data Types:
--------------------------------------------------------
{Data_Types_Cross_Check}

--------------------------------------------------------
Transaction Count:
--------------------------------------------------------
{Transaction_Count_Cross_Check}

--------------------------------------------------------
Summary Statistics by Transaction Type:
--------------------------------------------------------
{Summary_Statistics_Cross_Check}

========================================================
""")


              DATABASE CROSS VERIFICATION

Total Rows:
797885

--------------------------------------------------------
Data Types:
--------------------------------------------------------
Invoice            string[python]
StockCode          string[python]
Description        string[python]
Quantity                    Int64
InvoiceDate        datetime64[ns]
Price                     Float64
Customer ID                 Int64
Country            string[python]
Revenue                   Float64
Year                        int32
Month                      object
Month_Num                   int32
Day                        object
Day_Num                     int32
Quarter                    object
TransactionType            object
dtype: object

--------------------------------------------------------
Transaction Count:
--------------------------------------------------------
TransactionType
Sale            36975
cancellation     7901
Name: Invoice, dtype: int64

-----------------------------